# Binaire classificatie met de Digits dataset: **8 vs. de rest**

In dit notebook gebruiken we de `digits` dataset uit scikit-learn om een **binaire classifier** te trainen die onderscheid maakt tussen:

- **Klasse 1 (positief):** het cijfer **8**
- **Klasse 0 (negatief):** alle andere cijfers (0–7 en 9)

We gebruiken een **logistische regressie** als model en illustreren de belangrijkste evaluatieconcepten voor classificatie:

- Accuracy
- Precision
- Recall (True Positive Rate / sensitiviteit)
- F1-score
- Confusion matrix
- ROC-curve + AUC (Area Under the Curve)
- Precision–Recall curve

Het doel is om te laten zien dat dit géén “perfect” classificatieprobleem is: sommige 8’en lijken visueel sterk op andere cijfers, en omgekeerd. Daardoor zijn de ROC- en PR-curves realistischer dan bij de Breast Cancer dataset.


## 1. Data inladen en verkennen

We laden de `digits` dataset en bekijken kort de structuur.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

digits = load_digits()
X = digits.data      # 8x8 afbeeldingen als vector van lengte 64
y = digits.target    # cijferlabels 0 t/m 9

print("Vorm van X:", X.shape)
print("Unieke klassen in y:", np.unique(y))

### Voorbeeldafbeeldingen

We visualiseren een paar willekeurige voorbeelden om een gevoel te krijgen bij de data.


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
axes = axes.ravel()

for i in range(16):
    axes[i].imshow(digits.images[i], cmap='gray_r')
    axes[i].set_title(str(digits.target[i]))
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 2. Binaire labels maken: 8 vs. de rest

We maken van de multiclass labels een binaire target:

- `1` = het cijfer **8**
- `0` = alle overige cijfers

Daarna bekijken we de klassenbalans.


In [ ]:
# 1 = cijfer 8, 0 = alle andere cijfers
y_binary = (y == 8).astype(int)

unique, counts = np.unique(y_binary, return_counts=True)
class_counts = dict(zip(unique, counts))
print("Klassenverdeling (0 = niet-8, 1 = 8):", class_counts)

## 3. Train-test split en modeltraining

We splitsen de data in een trainings- en testset en trainen een logistische regressie.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

model = LogisticRegression(max_iter=1000, solver='liblinear')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # kans op klasse 1 (cijfer 8)

## 4. Basismaten: accuracy, precision, recall, F1-score

We beginnen met de standaard classificatiematen op basis van de **default threshold 0.5**:


In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1-score : {f1:.3f}")

### Interpretatie van de basismaten

- **Accuracy**: welk deel van alle testvoorbeelden (8 én niet-8) is correct geclassificeerd?  
- **Precision**: van alle voorspelde 8’en, hoeveel waren echt 8?  
- **Recall (TPR / sensitiviteit)**: van alle echte 8’en in de testset, hoeveel heeft het model gevonden?  
- **F1-score**: harmonisch gemiddelde van precision en recall; hoog alleen als beide goed zijn.

In deze setting is de klasse “8” relatief klein vergeleken met de rest. Daarom is **accuracy alleen niet voldoende**: je wilt vooral weten of het model **niet te veel 8’en mist (recall)** en of het **weinig valse alarmen geeft (precision)**.


## 5. Confusion matrix en foutentypen

We bekijken nu de confusion matrix voor de binaire classificatie (8 vs. rest).


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Niet-8', '8'])
ax.set_yticklabels(['Niet-8', '8'])
ax.set_xlabel('Voorspeld')
ax.set_ylabel('Werkelijk')
ax.set_title('Confusion matrix')

# Waarden in de vakjes plotten
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white')

plt.tight_layout()
plt.show()

### Interpretatie van de confusion matrix

De confusion matrix heeft de structuur:

|               | Voorspeld: 0 (niet-8) | Voorspeld: 1 (8) |
|---------------|------------------------|-------------------|
| **Werkelijk: 0 (niet-8)** | TN                      | FP                |
| **Werkelijk: 1 (8)**     | FN                      | TP                |

- **TP (True Positives)**: correcte herkenning van het cijfer 8  
- **FP (False Positives)**: model denkt dat het een 8 is, maar het is een ander cijfer  
- **FN (False Negatives)**: gemiste 8’en (model zegt niet-8)  
- **TN (True Negatives)**: correcte herkenning van niet-8

Voor een goed model wil je:

- zo min mogelijk **FN** (niet te veel 8’en missen → hoge recall)  
- zo min mogelijk **FP** (niet te vaak andere cijfers als 8 bestempelen → hoge precision)


## 6. ROC-curve en AUC

De ROC-curve laat de trade-off zien tussen:

- **True Positive Rate (TPR / recall)** op de y-as  
- **False Positive Rate (FPR)** op de x-as  

voor alle mogelijke thresholds.

Een perfecte classifier zou helemaal in de linkerbovenhoek zitten (AUC = 1.0). In realistische problemen ligt de AUC daar duidelijk onder.


In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

print(f"ROC AUC: {roc_auc:.3f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC-curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('ROC-curve')
plt.legend()
plt.tight_layout()
plt.show()

### Interpretatie ROC-curve

- De diagonale stippellijn stelt een **random classifier** voor (AUC = 0.5).  
- Hoe verder de ROC-curve **boven** deze lijn ligt, hoe beter het model onderscheid maakt tussen 8 en niet-8.  
- De AUC-waarde geeft dit in één getal weer: dichter bij 1.0 = beter.

Omdat dit probleem niet perfect lineair te scheiden is (sommige 8’en lijken sterk op andere cijfers), is de AUC **hoog maar niet perfect**. Dat is precies wat we willen zien in een realistisch classificatievoorbeeld.


## 7. Precision–Recall curve

De Precision–Recall (PR) curve is vooral nuttig bij **ongebalanceerde datasets**, waarbij de positieve klasse (hier: het cijfer 8) relatief zeldzaam is.


In [ ]:
precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

print(f"Average Precision (AP): {ap:.3f}")

plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions, label=f'PR-curve (AP = {ap:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall curve')
plt.legend()
plt.tight_layout()
plt.show()

### Interpretatie PR-curve

- Bij **lage recall** (we zijn heel streng) is de precision vaak hoog: we pakken dan alleen de meest zekere 8-voorspellingen.  
- Naarmate we **meer recall** willen (meer 8’en vinden), daalt de precision meestal: we pakken dan ook twijfelgevallen mee en krijgen meer false positives.

De PR-curve laat zien hoe goed het model presteert specifiek op de **positieve klasse (8)**. De average precision (AP) vat de curve samen in één getal (hoger = beter).


## 8. Samenvatting

In dit notebook hebben we een binaire classifier getraind om te bepalen of een cijfer **8** is of niet (8 vs. rest).  

We hebben de volgende concepten geïllustreerd:

- **Accuracy**: algemene juistheid van het model, maar niet voldoende bij scheve klassen.  
- **Precision**: hoe betrouwbaar een positieve voorspelling (8) is.  
- **Recall (TPR / sensitiviteit)**: hoeveel echte 8’en we daadwerkelijk vinden.  
- **F1-score**: balans tussen precision en recall.  
- **Confusion matrix**: inzicht in TP, FP, FN en TN.  
- **ROC-curve + AUC**: hoe goed het model positieven van negatieven kan scheiden over alle thresholds.  
- **Precision–Recall curve + AP**: prestaties gericht op de positieve klasse (8), vooral nuttig bij onbalans.

Belangrijk didactisch punt:  
De ROC- en PR-curves zijn **goed maar niet perfect** → dit is een realistisch scenario waarin het model het redelijk tot goed doet, maar fouten maakt op verwarrende cijfers. 
